# SPAM/NOT SPAM: Starter notebook
Copy (fork) and edit as many copies of this notebook as you require 

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 36)
pd.set_option("display.max_colwidth", 150)

seed = 42
import numpy as np
np.random.seed(seed)

# competition metric for local evaluation
from sklearn.metrics import matthews_corrcoef

## Read in the training data

In [2]:
train = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/train.csv", index_col = "row_id")
train

,text,spam_label
row_id,,
0,"You are everywhere dirt, on the floor, the windows, even on my shirt. And sometimes when i open my mouth, you are all that comes flowing out. I dr...",0
1,"Subject: \nmon , 24 may 2004 12 : 14 : 06 - 0600\ni am taking the liberty of writing you this\nletter instead of interrupting you\nby phone . plea...",1
2,So the sun is anti sleep medicine.,0
3,Hey are you angry with me. Reply me dr.,0
4,Ü go home liao? Ask dad to pick me up at 6...,0
...,...,...
7092,I stayed at this hotel over the weekend of the Chicago Bears Fan Convention (Feb 27- March 1). The hotel is beautiful. I had a Towers room. I had ...,0
7093,Subject: eis invoices for may\ni just wanted to make all of you aware of the message below from financial\noperations . when you review your rc re...,0
7094,Ok... The theory test? when are ü going to book? I think it's on 21 may. Coz thought wanna go out with jiayin. But she isnt free,0


# 📚 PROGRESIÓN DE MODELOS NLP - CLASIFICACIÓN SPAM/NOT SPAM

Este notebook contiene 4 modelos progresivos, cada uno con mejoras específicas.
Perfecto para preparar exámenes de IA donde debes:
- Completar arquitecturas
- Justificar decisiones de diseño
- Mejorar modelos existentes

---

## 🔧 Preparación de datos comunes para todos los modelos

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM, GRU, Dropout, LayerNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import tensorflow as tf

# Fijar semilla para reproducibilidad
tf.random.set_seed(seed)

# Separar datos de entrenamiento y validación
X = train['message'].values
y = train['spam_label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

print(f"📊 Datos de entrenamiento: {len(X_train)}")
print(f"📊 Datos de validación: {len(X_val)}")
print(f"📊 Datos de test: {len(test)}")

In [ ]:
# Configuración de tokenización y padding
MAX_WORDS = 5000  # Vocabulario máximo
MAX_LEN = 100     # Longitud máxima de secuencia

# Tokenizar textos
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convertir textos a secuencias
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(test['message'].values)

# Padding
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"✅ Tokenización completa")
print(f"Vocabulario: {len(tokenizer.word_index)} palabras")
print(f"Shape entrenamiento: {X_train_pad.shape}")
print(f"Shape validación: {X_val_pad.shape}")

---

## 🟢 MODELO 1: Red Neuronal Básica (Baseline)

### 📋 Características:
- **Embedding layer**: Convierte palabras en vectores densos
- **Flatten/GlobalAveragePooling**: Reduce dimensionalidad
- **Dense layer**: Clasificación binaria con sigmoid

### ⚠️ Problema esperado:
Este modelo **NO captura el orden de las palabras**. Trata el texto como una "bolsa de palabras" (Bag of Words).
Para spam, el orden puede ser importante ("not spam" vs "spam not").

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling1D

# MODELO 1: Baseline simple
model_1 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=32, input_length=MAX_LEN),
    GlobalAveragePooling1D(),  # Promedia todos los embeddings (pierde orden)
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # Clasificación binaria
])

model_1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_1.summary()

In [ ]:
# Entrenar Modelo 1
print("🚀 Entrenando MODELO 1 - Baseline...")
history_1 = model_1.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

# Evaluar
val_loss_1, val_acc_1 = model_1.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 MODELO 1 - Val Accuracy: {val_acc_1:.4f}")

---

## 🔵 MODELO 2: Añadir LSTM (Captura Secuencias)

### 📋 Mejora respecto al Modelo 1:
- **Problema del modelo 1**: GlobalAveragePooling pierde el orden de las palabras
- **Solución**: Usar **LSTM** (Long Short-Term Memory)
  
### 🧠 ¿Por qué LSTM?
- LSTM procesa secuencias **en orden**
- Tiene **memoria a largo plazo** (celda de estado)
- Puede recordar contexto: "not spam" ≠ "spam not"

### 📚 Teoría de examen:
> "Las RNN/LSTM son necesarias cuando el **orden temporal/secuencial** importa.
> En NLP, el orden de las palabras cambia el significado."

In [ ]:
# MODELO 2: Con LSTM
model_2 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    LSTM(64),  # ✨ MEJORA: Captura orden de palabras
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_2.summary()

In [ ]:
# Entrenar Modelo 2
print("🚀 Entrenando MODELO 2 - Con LSTM...")
history_2 = model_2.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

# Evaluar
val_loss_2, val_acc_2 = model_2.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 MODELO 2 - Val Accuracy: {val_acc_2:.4f}")
print(f"📈 Mejora respecto Modelo 1: {(val_acc_2 - val_acc_1)*100:.2f}%")

---

## 🟠 MODELO 3: LSTM + Dropout (Regularización)

### 📋 Mejora respecto al Modelo 2:
- **Problema del modelo 2**: Puede sufrir **overfitting** (sobreajuste)
  - El modelo memoriza los datos de entrenamiento
  - No generaliza bien a datos nuevos
  
- **Solución**: Añadir **Dropout**

### 🧠 ¿Por qué Dropout?
- Dropout "apaga" aleatoriamente neuronas durante entrenamiento
- Fuerza al modelo a aprender representaciones **robustas**
- Reduce la dependencia de neuronas específicas
- Mejora la **generalización**

### 📚 Teoría de examen:
> "Dropout es una técnica de **regularización** que previene overfitting.
> Típicamente se usa con valores entre 0.2-0.5 (20%-50% de neuronas apagadas)."

In [ ]:
# MODELO 3: LSTM + Dropout
model_3 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    Dropout(0.3),  # ✨ MEJORA 1: Dropout después del embedding
    LSTM(64),
    Dropout(0.4),  # ✨ MEJORA 2: Dropout después del LSTM
    Dense(16, activation='relu'),
    Dropout(0.3),  # ✨ MEJORA 3: Dropout antes de la capa final
    Dense(1, activation='sigmoid')
])

model_3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_3.summary()

In [ ]:
# Entrenar Modelo 3
print("🚀 Entrenando MODELO 3 - LSTM + Dropout...")
history_3 = model_3.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,  # Más épocas porque Dropout hace el entrenamiento más lento
    batch_size=32,
    verbose=1
)

# Evaluar
val_loss_3, val_acc_3 = model_3.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 MODELO 3 - Val Accuracy: {val_acc_3:.4f}")
print(f"📈 Mejora respecto Modelo 2: {(val_acc_3 - val_acc_2)*100:.2f}%")

---

## 🟣 MODELO 4: Bidirectional LSTM + Layer Normalization + Early Stopping

### 📋 Mejoras respecto al Modelo 3:

#### 1️⃣ **Bidirectional LSTM**
- **Problema del LSTM unidireccional**: Solo lee de izquierda → derecha
- **Solución**: Bidirectional lee en **ambas direcciones**
- **Ventaja en NLP**: Captura contexto previo Y posterior
  - Ejemplo: "The movie was not _____" → necesita contexto de ambos lados

#### 2️⃣ **Layer Normalization**
- **Problema**: Las activaciones pueden tener escalas muy diferentes
- **Solución**: LayerNormalization normaliza las activaciones
- **Ventaja**: Estabiliza el entrenamiento, acelera convergencia

#### 3️⃣ **Early Stopping**
- **Problema**: No sabemos cuántas épocas son óptimas
- **Solución**: Para el entrenamiento cuando val_loss deja de mejorar
- **Ventaja**: Evita overfitting automáticamente

### 📚 Teoría de examen:
> "Bidirectional LSTM procesa secuencias en **ambas direcciones** para capturar contexto completo.
> Layer Normalization estabiliza activaciones, mejorando velocidad y estabilidad de entrenamiento.
> Early Stopping es un callback que previene overfitting automáticamente."

In [ ]:
# MODELO 4: Bidirectional LSTM + LayerNorm + Early Stopping
model_4 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    Dropout(0.3),
    
    # ✨ MEJORA 1: Bidirectional LSTM (lee en ambas direcciones)
    Bidirectional(LSTM(64, return_sequences=True)),
    LayerNormalization(),  # ✨ MEJORA 2: Normalización de capa
    Dropout(0.4),
    
    # Segunda capa LSTM (más profundidad)
    Bidirectional(LSTM(32)),
    LayerNormalization(),
    Dropout(0.4),
    
    # Capas densas finales
    Dense(32, activation='relu'),
    LayerNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_4.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_4.summary()

In [ ]:
# ✨ MEJORA 3: Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',      # Monitorear pérdida de validación
    patience=3,              # Esperar 3 épocas sin mejora
    restore_best_weights=True,  # Restaurar los mejores pesos
    verbose=1
)

# Entrenar Modelo 4
print("🚀 Entrenando MODELO 4 - Bidirectional LSTM + LayerNorm + EarlyStopping...")
history_4 = model_4.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=30,  # Muchas épocas, pero EarlyStopping parará antes
    batch_size=32,
    callbacks=[early_stop],  # ⚡ Callback de early stopping
    verbose=1
)

# Evaluar
val_loss_4, val_acc_4 = model_4.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 MODELO 4 - Val Accuracy: {val_acc_4:.4f}")
print(f"📈 Mejora respecto Modelo 3: {(val_acc_4 - val_acc_3)*100:.2f}%")

---

## 🎯 MODELO 5 (BONUS): GRU como alternativa a LSTM

### 📋 Mejora respecto al Modelo 4:
- **Problema del LSTM**: Tiene muchos parámetros (4 gates: input, forget, output, cell)
- **Solución**: Usar **GRU** (Gated Recurrent Unit)

### 🧠 ¿Por qué GRU?
- **Menos parámetros** que LSTM (solo 2 gates: reset, update)
- **Más rápido** de entrenar
- **Rendimiento similar** en muchas tareas
- Mejor cuando hay **menos datos** o se quiere velocidad

### 📚 Diferencia LSTM vs GRU:
| Característica | LSTM | GRU |
|---|---|---|
| **Gates** | 4 (input, forget, output, cell) | 2 (reset, update) |
| **Parámetros** | Más | Menos (~75% del LSTM) |
| **Velocidad** | Más lento | Más rápido |
| **Capacidad** | Mayor memoria a largo plazo | Suficiente para la mayoría de casos |
| **Cuándo usar** | Secuencias muy largas, mucha data | Menos data, necesitas velocidad |

In [ ]:
# MODELO 5: Bidirectional GRU (alternativa más ligera)
model_5 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    Dropout(0.3),
    
    # ✨ CAMBIO: GRU en lugar de LSTM (menos parámetros, más rápido)
    Bidirectional(GRU(64, return_sequences=True)),
    LayerNormalization(),
    Dropout(0.4),
    
    Bidirectional(GRU(32)),
    LayerNormalization(),
    Dropout(0.4),
    
    Dense(32, activation='relu'),
    LayerNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model_5.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_5.summary()

In [ ]:
# Entrenar Modelo 5
print("🚀 Entrenando MODELO 5 - Bidirectional GRU...")
history_5 = model_5.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluar
val_loss_5, val_acc_5 = model_5.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 MODELO 5 - Val Accuracy: {val_acc_5:.4f}")
print(f"⚡ GRU vs LSTM - Diferencia: {(val_acc_5 - val_acc_4)*100:.2f}%")

# Comparar número de parámetros
print(f"\n🔢 Parámetros LSTM (Modelo 4): {model_4.count_params():,}")
print(f"🔢 Parámetros GRU (Modelo 5): {model_5.count_params():,}")
print(f"💾 Reducción: {(1 - model_5.count_params()/model_4.count_params())*100:.1f}%")

---

## 📊 COMPARACIÓN FINAL DE TODOS LOS MODELOS

In [ ]:
import matplotlib.pyplot as plt

# Crear comparación visual
models_names = ['Modelo 1\n(Baseline)', 'Modelo 2\n(LSTM)', 'Modelo 3\n(+Dropout)', 
                'Modelo 4\n(BiLSTM+LN)', 'Modelo 5\n(BiGRU)']
accuracies = [val_acc_1, val_acc_2, val_acc_3, val_acc_4, val_acc_5]

plt.figure(figsize=(12, 6))
bars = plt.bar(models_names, accuracies, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57'])
plt.ylabel('Validation Accuracy', fontsize=12)
plt.xlabel('Modelos', fontsize=12)
plt.title('🎯 Comparación de Accuracy de Validación entre Modelos', fontsize=14, fontweight='bold')
plt.ylim([min(accuracies) - 0.02, max(accuracies) + 0.02])
plt.grid(axis='y', alpha=0.3)

# Añadir valores encima de las barras
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{acc:.4f}',
             ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# Tabla comparativa
print("\n" + "="*80)
print("📋 TABLA COMPARATIVA DE MODELOS".center(80))
print("="*80)
for i, (name, acc) in enumerate(zip(models_names, accuracies), 1):
    improvement = ((acc - accuracies[i-2])*100) if i > 1 else 0
    print(f"Modelo {i}: {name.replace(chr(10), ' '):<30} Accuracy: {acc:.4f}  (+{improvement:>5.2f}%)")
print("="*80)

---

# 📚 RESUMEN TIPO EXAMEN - CHEAT SHEET

## 🎓 Progresión de Modelos y Justificaciones

### **MODELO 1: Baseline (Red Densa Simple)**
- **Arquitectura**: Embedding → GlobalAveragePooling → Dense
- **Problema detectado**: ❌ No captura el ORDEN de las palabras (Bag of Words)
- **Limitación**: "not spam" = "spam not" (trata igual ambas frases)

---

### **MODELO 2: LSTM**
- **Mejora aplicada**: ✅ Añadir capa LSTM
- **Justificación**: LSTM procesa secuencias en orden, tiene memoria a largo plazo
- **Efecto esperado**: Mejor comprensión de contexto secuencial
- **Cuándo usarlo**: Cuando el orden temporal/secuencial importa

---

### **MODELO 3: LSTM + Dropout**
- **Problema del modelo 2**: ❌ Overfitting (memoriza datos de entrenamiento)
- **Mejora aplicada**: ✅ Añadir Dropout (0.3-0.4) en varias capas
- **Justificación**: Dropout apaga neuronas aleatoriamente → fuerza robustez
- **Efecto esperado**: Mejor generalización a datos nuevos

---

### **MODELO 4: Bidirectional LSTM + LayerNormalization + EarlyStopping**
- **Problema del modelo 3**: ❌ LSTM solo lee en una dirección, activaciones inestables
- **Mejoras aplicadas**:
  1. ✅ **Bidirectional**: Lee izquierda→derecha Y derecha→izquierda
  2. ✅ **LayerNormalization**: Normaliza activaciones para estabilidad
  3. ✅ **EarlyStopping**: Para entrenamiento automáticamente cuando val_loss no mejora
- **Efecto esperado**: Mejor contexto completo, entrenamiento más estable

---

### **MODELO 5: GRU (alternativa a LSTM)**
- **Problema del LSTM**: ❌ Muchos parámetros (4 gates) → lento, necesita más datos
- **Mejora aplicada**: ✅ Cambiar LSTM → GRU (solo 2 gates)
- **Justificación**: 
  - GRU tiene ~25% menos parámetros
  - Más rápido de entrenar
  - Rendimiento similar en muchas tareas
- **Cuándo usar GRU**: Menos datos, necesitas velocidad, secuencias no muy largas
- **Cuándo usar LSTM**: Secuencias muy largas, muchos datos, máxima capacidad

---

## 🔑 CONCEPTOS CLAVE PARA EXAMEN

### **1. ¿Por qué usar capas recurrentes (LSTM/GRU)?**
> Respuesta: Para capturar **dependencias secuenciales** y **orden temporal** en los datos.
> En NLP, el orden de las palabras cambia el significado.

### **2. ¿Qué es Dropout y por qué se usa?**
> Respuesta: Técnica de **regularización** que apaga aleatoriamente neuronas (20-50%).
> Previene **overfitting** y mejora **generalización**.

### **3. ¿Diferencia entre LSTM y GRU?**
> - **LSTM**: 4 gates, más parámetros, mayor capacidad de memoria
> - **GRU**: 2 gates, más rápido, suficiente para la mayoría de casos
> - **Trade-off**: Capacidad vs Velocidad

### **4. ¿Qué es Bidirectional?**
> Respuesta: Procesa secuencia en **ambas direcciones** (forward + backward).
> Captura contexto completo (antes Y después de cada palabra).

### **5. ¿Qué es Layer Normalization?**
> Respuesta: Normaliza activaciones de una capa para tener media 0 y varianza 1.
> Estabiliza entrenamiento y acelera convergencia.

### **6. ¿Qué es Early Stopping?**
> Respuesta: Callback que para el entrenamiento cuando una métrica (e.g., val_loss) 
> deja de mejorar por N épocas (patience). Restaura los mejores pesos.

---

## 🎯 PREGUNTAS TÍPICAS DE EXAMEN

**P1: "El modelo tiene baja accuracy en validación pero alta en train. ¿Qué hacer?"**
> R: Overfitting. Soluciones: Dropout, Early Stopping, más datos, regularización L2.

**P2: "Necesito capturar contexto de palabras anteriores y posteriores. ¿Qué usar?"**
> R: Bidirectional LSTM o Bidirectional GRU.

**P3: "El entrenamiento es inestable (loss sube y baja mucho). ¿Qué hacer?"**
> R: Usar Layer Normalization o Batch Normalization. Reducir learning rate.

**P4: "Tengo pocos datos. ¿LSTM o GRU?"**
> R: GRU. Menos parámetros → menos propenso a overfitting con pocos datos.

**P5: "¿Dónde poner Dropout en un modelo NLP?"**
> R: Después de Embedding, después de capas recurrentes, antes de capa de salida.

---

## 📐 ARQUITECTURA TÍPICA NLP (PLANTILLA)

```python
model = Sequential([
    # 1. EMBEDDING: Convierte palabras → vectores
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Dropout(0.2-0.3),
    
    # 2. CAPA RECURRENTE: Procesa secuencias
    Bidirectional(LSTM(units, return_sequences=True)),  # Si hay más capas después
    # O Bidirectional(LSTM(units)) si es la última capa recurrente
    LayerNormalization(),  # Opcional pero recomendado
    Dropout(0.3-0.5),
    
    # 3. SEGUNDA CAPA RECURRENTE (opcional, más profundidad)
    Bidirectional(LSTM(units//2)),
    LayerNormalization(),
    Dropout(0.3-0.5),
    
    # 4. CAPAS DENSAS: Clasificación
    Dense(units, activation='relu'),
    Dropout(0.2-0.3),
    Dense(1, activation='sigmoid')  # Binaria
    # O Dense(num_classes, activation='softmax')  # Multiclase
])
```

---

## ⚠️ ERRORES COMUNES EN EXÁMENES

1. ❌ Usar `return_sequences=True` en la última capa LSTM → ERROR: shape incompatible con Dense
2. ❌ No usar `input_length` en Embedding → Puede causar problemas
3. ❌ Olvidar `activation='sigmoid'` en clasificación binaria
4. ❌ Usar `accuracy` en lugar de `binary_accuracy` para clasificación binaria
5. ❌ No compilar el modelo antes de entrenar
6. ❌ No normalizar/tokenizar el texto antes de pasarlo al modelo

---

## 🏆 TABLA DE DECISIÓN RÁPIDA

| Situación | Solución |
|-----------|----------|
| Overfitting | Dropout, Early Stopping, L2 regularization |
| Underfitting | Más capas, más units, entrenar más tiempo |
| Entrenamiento lento | GRU en vez de LSTM, menos layers, batch size mayor |
| Entrenamiento inestable | Layer Normalization, learning rate menor |
| Necesito contexto completo | Bidirectional LSTM/GRU |
| Pocos datos | GRU, Dropout alto, Data Augmentation |
| Secuencias muy largas | LSTM (mejor memoria) |
| No importa el orden | GlobalAveragePooling, CNN 1D |

---

## 🚀 SELECCIONAR EL MEJOR MODELO Y HACER PREDICCIONES

Ahora vamos a usar el mejor modelo (probablemente el Modelo 4 o 5) para hacer predicciones en el test set.

In [ ]:
# Seleccionar el mejor modelo basado en val_accuracy
best_model_idx = accuracies.index(max(accuracies))
best_model_name = models_names[best_model_idx].replace('\n', ' ')
best_models = [model_1, model_2, model_3, model_4, model_5]
best_model = best_models[best_model_idx]

print(f"🏆 MEJOR MODELO: {best_model_name}")
print(f"📊 Validation Accuracy: {max(accuracies):.4f}")
print(f"\n🔮 Generando predicciones en test set...")

---

## 🎓 EJERCICIOS ADICIONALES PARA PRACTICAR

Intenta estas modificaciones para practicar para el examen:

### **Ejercicio 1: Añadir una tercera capa LSTM**
```python
# Modifica el Modelo 4 para añadir una tercera capa Bidirectional LSTM
# Recuerda: return_sequences=True en las capas intermedias
```

### **Ejercicio 2: Usar diferentes optimizadores**
```python
# Prueba con: 'adam', 'rmsprop', 'sgd'
# ¿Cuál funciona mejor?
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
```

### **Ejercicio 3: Experimentar con Embedding dimension**
```python
# Prueba con: 32, 64, 128, 256
# ¿Más dimensiones = mejor rendimiento? ¿Siempre?
Embedding(input_dim=MAX_WORDS, output_dim=256, input_length=MAX_LEN)
```

### **Ejercicio 4: Añadir Batch Normalization en lugar de Layer Normalization**
```python
from tensorflow.keras.layers import BatchNormalization
# ¿Cuál funciona mejor? ¿Por qué?
```

### **Ejercicio 5: Combinar CNN + LSTM**
```python
from tensorflow.keras.layers import Conv1D, MaxPooling1D
# CNN captura patrones locales, LSTM captura secuencias
# Arquitectura: Embedding → Conv1D → MaxPooling → LSTM → Dense
```

---

## 🎯 CONSEJOS FINALES PARA EL EXAMEN

### ✅ **Lo que SÍ debes saber**:
1. **Orden de capas**: Embedding → Recurrente → Dense
2. **return_sequences**: True si hay más capas después, False si es la última recurrente
3. **Activaciones**: sigmoid (binaria), softmax (multiclase), relu (capas ocultas)
4. **Justificaciones**: Por qué LSTM > Bag of Words, por qué Dropout, por qué Bidirectional
5. **Trade-offs**: LSTM vs GRU, profundidad vs overfitting, capacidad vs velocidad

### ❌ **Errores que debes evitar**:
1. Olvidar compilar el modelo antes de entrenar
2. No usar `input_length` en Embedding
3. Mezclar shapes incompatibles (return_sequences)
4. No normalizar/tokenizar los datos
5. Usar demasiadas capas con pocos datos (overfitting seguro)

### 📝 **Si el examen pide "completar una arquitectura"**:
- Fíjate en los shapes: (batch, timesteps, features)
- Si ves (None, timesteps, X) → necesitas capa recurrente o flatten
- Si ves (None, X) → necesitas capa Dense
- Si return_sequences=True → hay más capas después
- Si es la última recurrente → return_sequences=False (default)

### 💡 **Si el examen pide "mejorar un modelo"**:
1. Identifica el problema: ¿Overfitting? ¿No captura orden? ¿Lento?
2. Aplica la solución correspondiente:
   - Overfitting → Dropout, Early Stopping
   - No captura orden → LSTM/GRU
   - Lento → GRU en vez de LSTM, menos capas
   - Necesita contexto completo → Bidirectional
3. Justifica tu decisión con teoría

---

## 🎊 ¡Buena suerte en el examen!

In [ ]:
# Hacer predicciones en el test set con el mejor modelo
y_pred_proba = best_model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

print(f"✅ Predicciones generadas: {len(y_pred)}")
print(f"📊 Distribución: {sum(y_pred)} spam, {len(y_pred) - sum(y_pred)} not spam")

## This is the test data that you are asked to make predictions for

In [3]:
test = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/test.csv", index_col = "row_id")
test

,text
row_id,
7097,"Subject: king ranch processed volumes at tailgate\nd .\nmary provided this to brian , please check against your numbers asap\n- - - - - - - - - - ..."
7098,"Subject: cialis , viagra , xanax , valium at low price ! no prescription needed !\ndiscount rx is simple , quick , and affordable ! br\noffering m..."
7099,"Subject: september deal inactivation in sitara\ncurrently , prior month deals are inactivated in sitara for portfolio\npurposes on the 10 th of th..."
7100,Subject: jan . 01 sale to texas general land office\nlinda -\ni did not know that this was part of a natural gas / crude oil exchange deal\nand bi...
7101,Subject: 5 th changes @ duke and air liquide\n- - - - - - - - - - - - - - - - - - - - - - forwarded by ami chokshi / corp / enron on 02 / 04 / 200...
...,...
11824,"Subject: contract obligations\ncharlie ,\nhere is a breakout for the volumes that have been paid on . we are off by\napproximately 20000 according..."
11825,"For a hotel rated with four diamonds by AAA, one would think the Hilton Chicago would be almost like staying at a palace with royalty. The only ro..."
11826,"We spend our days waiting for the ideal path to appear in front of us.. But what we forget is.. ""paths are made by walking.. not by waiting.."" Goo..."


In [4]:
# baseline model predictions
y_pred = 0

## Submit your predictions in a `submission.csv` file for scoring on the [leaderboard](https://www.kaggle.com/competitions/u-tad-spam-not-spam-2025-edition/leaderboard)
To submit your notebook click on **Submit to competition** and then **Submit**.

In [5]:
# do not modify this code
submission = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/sample_submission.csv")
submission["spam_label"] = y_pred
submission.to_csv('submission.csv',index=False)

In [6]:
submission.head()

,row_id,spam_label
0,7097,0
1,7098,0
2,7099,0
3,7100,0
4,7101,0
